# Diffusion Tensor Imaging Measurements for Computer-Aided Diagnosis of Autism - Derived Data from ABIDE II Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process a Croissant-schema dataset using the `mlcroissant` library.

### Dataset Source
The dataset Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.dbrh-5zc8/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -q mlcroissant

## 1. Data Loading

Load metadata and data records from the FAIR² Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.dbrh-5zc8/fair2.json"

# Load the croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Published: {metadata.datePublished}, Version: {metadata.version}")

print(f"License: {metadata.license}\n")

print(f"Croissant @id: {metadata.id}")

## 2. Data Overview

Let's investigate what record sets, fields, and columns are available. **All identifiers use their Croissant `@id` fields.**

In [ ]:
# Listing all record sets and their fields

record_sets = dataset.record_sets
print("Available Record Sets and their @id fields:")
for rs in record_sets:
    print(f"- RecordSet Name: {rs.name} | @id: {rs.id}")
    print(f"   Fields (with @id):")
    for f in rs.fields:
        print(f"     - {f.name}: {f.id} (dataType: {f.data_type})")
    print('')

## 3. Data Extraction

**Load the data from one or more record sets into pandas DataFrames.**

We'll use the record set and field `@id` values identified above.

In [ ]:
# Pick all record set @ids (they are likely to be
# something like 'cr:RecordSet/0' or a similar URI)
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    # Load all records for this set
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Print the columns for the first available record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"RecordSet @id: {first_rs_id}")
    print("Columns (fields):", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)

Let's filter, normalize, and group the data using a numeric field. Make sure to use the correct field `@id` as the column name, referencing the output above.

- **Filtering**: Keep only records where a chosen numeric field exceeds a threshold.
- **Normalization**: Z-score normalize the selected field.
- **Grouping**: Aggregate by another key field if available.

In [ ]:
# Select your main record set and a numeric field by '@id',
# as shown in the overview above. We'll use the first set for illustration.

if record_set_ids:
    record_set_id = first_rs_id
    df = dataframes[record_set_id]

    # Look for a plausible numeric field
    numeric_field = None
    for field in dataset.record_set(record_set_id).fields:
        if field.data_type in ('Float', 'Integer', 'Number'):
            numeric_field = field.id
            break

    if numeric_field is not None and numeric_field in df.columns:
        # Choose a filtering threshold
        q = df[numeric_field].quantile(0.9) if df[numeric_field].dtype.kind in 'fiu' else 10
        filtered_df = df[df[numeric_field] > q]
        print(f"Filtered ({numeric_field} > {q}): {len(filtered_df)} records")

        # Normalize (Z-score)
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another field (prefer non-numeric, e.g., subject id or region)
        group_field = None
        for field in dataset.record_set(record_set_id).fields:
            if field.data_type not in ('Float', 'Integer', 'Number'):
                if field.id in df.columns:
                    group_field = field.id
                    break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).to_frame()
            print(f"\nMeans of {numeric_field} grouped by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No suitable numeric field found for analysis.")
else:
    print("No record sets available for EDA.")

## 5. Visualization

Let's visualize the distribution of a numeric field, e.g., fractional anisotropy or diffusivity metric, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field in dataframes[record_set_id].columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[record_set_id][numeric_field], bins=40, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

## 6. Conclusion

In this notebook, we've loaded the FAIR² DTI dataset, reviewed its metadata, explored schema-defined record sets and fields via their Croissant `@id`s, extracted data into DataFrames, and performed simple exploratory analyses using `mlcroissant`. You can now extend this analysis for further statistical modeling, machine learning, or neuroscience research.